# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [280]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [281]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [282]:
dirty = df.copy()

df = df.drop_duplicates()

before = len(dirty)

after = len(df)
log('drop duplicates', 'rows removed', f'{before - after}')

[drop duplicates] row(s) removed (15 row(s))


### TODO 2 — clean `price` -> float

In [283]:
dirty = df.copy()

df['price'] = df['price'].str.replace('$', '').astype(float)

changed_rows = dirty['price'].str.contains(r'\$', regex=True).sum()

log('clean price', 'cleaned price', changed_rows)


[clean price] cleaned price (124 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [284]:
dirty = df.copy()

df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
df = df.dropna(subset=['qty'])
df = df[df['qty'] > 0]
log('clean qty', 'cleaned qty', len(dirty) - len(df))


[clean qty] cleaned qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [285]:
dirty = df.copy()
ITEM_MAP = {
    'foam finger': 'Foam Finger',
    'Foam Finger': 'Foam Finger',
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'rain poncho': 'Rain Poncho',
    'Rain Poncho': 'Rain Poncho'}
df['item'] = df['item'].map(ITEM_MAP)
changed_rows = (dirty['item'] != df['item']).sum()
print(df['item'].value_counts())
log('canonicalize item', 'canonicalized item', changed_rows)



item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[canonicalize item] canonicalized item (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [286]:
dirty = df.copy()
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Apparel',
    'rain-gear': 'RainGear',
    'RainGear': 'RainGear',
    'Apparel': 'Apparel'}
df['category'] = df['category'].map(CATEGORY_MAP)
changed_rows = (dirty['category'] != df['category']).sum()
print(df['category'].value_counts())
log('canonicalize category', 'canonicalized category', changed_rows)


category
Food        95
Apparel     94
RainGear    86
Name: count, dtype: int64
[canonicalize category] canonicalized category (140 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [287]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(ITEM_MAP.values()).all()
assert df['category'].nunique() == 3
print('clean:', df.shape)
df.head()

clean: (275, 5)


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,Foam Finger,Apparel,1.0,7.5
2,2,Cheeseburger,Apparel,1.0,7.5
4,4,Cheeseburger,Apparel,1.0,7.5
5,5,Foam Finger,Food,3.0,6.0


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [288]:
dirty = df.copy()
df['revenue'] = df['price']* df['qty']
revenue_by_category = (
    df.groupby('category')['revenue']
    .sum()
    .round(2)
    .sort_values(ascending=False)
)

print(revenue_by_category)
print('Total:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Apparel     1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total: 4740.0


**What I would tell the vendor: I would tell the vendor to stock more food, because it generates the most revenue.

### TODO 8 — read back your log

In [289]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop duplicates,row(s) removed,15
1,clean price,cleaned price,124
2,clean qty,cleaned qty,25
3,canonicalize item,canonicalized item,126
4,canonicalize category,canonicalized category,140


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

A. Cleaning quantity changed total revenue across all categories and items the most. Before it was 4594.5 and after it was 4740.0.

B. When I was normalalizing category I decided to combine merch and apparel in to  simply apparel. A reasonable person could have decided to keep them seperate maybe for data analysis purposes or  due to the fact merch ususally has a certain team, player, or celebrity attached. It could be reasonable to want to keep track of how much merch with famous individuals sell. This would have split my total apparel catgeory up and decreased the revenue it currently claims.